In [ ]:
#Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

## EDA and Sensor Analysis

In [ ]:
# Set column names
operational_settings = [f'operational_setting_{i}' for i in range(1, 4)]
sensor_measurements = [f'sensor_measurement_{i}' for i in range(1, 22)]
cols = ['unit_number', 'time_in_cycles'] + operational_settings + sensor_measurements

In [ ]:
# Load data
path_1 = 'dataset/train_FD001.txt'
train_001 = pd.read_csv(path_1, sep=r'\s+', header=None, names=cols)
train_001.head()

In [ ]:
# Compute Remaining Useful Life (RUL)
train_001['max_cycle'] = train_001.groupby('unit_number')['time_in_cycles'].transform('max')
train_001['RUL'] = train_001['max_cycle'] - train_001['time_in_cycles']

In [ ]:
# Plot sensor_measurements against RUL
#Independent sample of 5 engines
sample_units = train_001['unit_number'].unique()[:5]

fig, ax = plt.subplots(7, 3, figsize=(15, 20), sharex=True)
ax = ax.flatten()
for i, sensor in enumerate(sensor_measurements):
    for unit in sample_units:
        unit_df = train_001[train_001['unit_number']==unit]
        ax[i].plot(unit_df['RUL'], unit_df[sensor], label=f'eng{unit}', alpha=0.6)
    ax[i].set_ylabel('Sensor Measurement')
    ax[i].set_title(sensor, fontsize=9)
    ax[i].legend()
    ax[i].grid(True)
ax[0].invert_xaxis()
plt.tight_layout()
plt.show();

### Correlation with RUL
Correlation with RUL. Sensors 11, 4, 15, 2, 17, 3, 8 and 13 correlate negatively with RUL (down to about −0.70), and 12, 7, 21 and 20 positively (up to about 0.67). Sensors 9, 14 and 6 are weak. Constant sensors give NaN, meaning no signal.

In [ ]:
# Check each sensors correlation with RUL
train_001[sensor_measurements + ['RUL']].corr()['RUL'].sort_values()

In [ ]:
# Max cycle check for late-onset degradation sensors 9 and 14
# Compare max cycles of specific engines
train_001.groupby('unit_number')['time_in_cycles'].max().loc[[9,11, 12, 13, 14, 15]]

## Data Preparation
* Create a list of clear-degradation snesors (drop the flat ones)
* Create 'max_cycle' and 'RUL' columns for test set

In [ ]:
# Flat sensors
flat_numbers = [1, 5, 6, 10, 16, 18, 19]
flat_sensors = [f'sensor_measurement_{i}' for i in flat_numbers]

# Feature sensors
feature_sensors = [s for s in sensor_measurements if s not in flat_sensors]

### Test set

In [ ]:
path_2 = 'dataset/test_FD001.txt'
test_001 = pd.read_csv(path_2, sep=r'\s+', header=None, names=cols)
test_001.head()

In [ ]:
path_3 = 'dataset/RUL_FD001.txt'
RUL_001 = pd.read_csv(path_3, sep=r'\s+', header=None, names=['RUL'])
RUL_001['unit_number'] = RUL_001.index + 1
RUL_001

In [ ]:
# Create a dataframe containing ['unit_number' and 'time_in_cycles'],from the test_set
max_cycle_test = test_001.groupby('unit_number')['time_in_cycles'].max().reset_index()

# Merge RUL_001 and max_cycle_test on 'unit_number'
max_cycle_test = max_cycle_test.merge(RUL_001, on='unit_number')

# Add 'time_in_cycles' and 'RUL' to get the 'max_cycle'
max_cycle_test['max_cycle'] = max_cycle_test['time_in_cycles'] + max_cycle_test['RUL']

In [ ]:
# Merge max_cycle_test[['unit_number', 'max_cycle']] to test_001
test_001 = test_001.merge(max_cycle_test[['unit_number', 'max_cycle']], on='unit_number')

# Add the 'RUL' column like you did for training set
test_001['RUL'] = test_001['max_cycle'] - test_001['time_in_cycles']

## Evaluation Protocol
**Features and targets.** X is the 14 clear-degradation sensors (`feature_sensors`). Each model is trained on one of two targets: the raw `RUL`, or `RUL` capped at 125 cycles.

**Four configurations** (2 models x 2 targets):
* `linreg` (uncapped / capped): linear regression on standardised features. Trained on the uncapped target it produces physically impossible negative RUL predictions near end of life, consistent with the non-linear degradation seen in the sensor analysis.
* `rf` (uncapped / capped): random forest. Tree-based models are bounded by their training targets, so they cannot predict negative RUL.
* *Why cap?* The sensors show little or no degradation signal early in an engine's life, so predicting a high RUL from them is guesswork. 125 is the convention from the CMAPSS literature (Heimes, 2008), not a physical constant.

**Model selection: grouped validation (training engines only).** `GroupKFold(5)` with `unit_number` as the group, so no engine appears in both the training and validation parts of a fold (neighbouring cycles of one engine are near-duplicates). The scaler sits inside a pipeline so it is fit on the training part of each fold only. All configurations are scored on the same validation rows: those with true RUL <= 125, where capped and uncapped targets coincide. This is a fair way to *compare* configurations, but it does not test early-life behaviour.

**Final evaluation: test set, once.** Configurations are retrained on all 100 training engines and scored on the last observed cycle of each of the 100 test engines against `RUL_FD001.txt` (the official CMAPSS protocol).

In [ ]:
cap = 125

In [ ]:
def make_models():
    """ Unfitted models on every call, so no fold inherits anything from another"""
    return {
        'linreg': make_pipeline(StandardScaler(), LinearRegression()),
        'rf': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    }

In [ ]:
def run_cv(feature_cols, n_splits=5):
    X = train_001[feature_cols]
    y_true = train_001['RUL']
    y_cap = y_true.clip(upper=cap)
    groups = train_001['unit_number']

    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    for fold, (tr, va) in enumerate(gkf.split(X, y_true, groups), start=1):
        #check that no engine appears twice on both sides of the split
        assert set(groups.iloc[tr]).isdisjoint(groups.iloc[va]) 

        y_va = y_true.iloc[va].values
        in_range = y_va<=cap

        for target_name, y_tr in [('uncapped', y_true.iloc[tr]), ('capped', y_cap.iloc[tr])]:
            for model_name, model in make_models().items():
                model.fit(X.iloc[tr], y_tr)
                preds = model.predict(X.iloc[va])
                rmse= np.sqrt (mean_squared_error(y_va[in_range], preds[in_range]))
                rows.append({'fold':fold, 'model':model_name, 'trained_on':target_name, 'rmse':rmse})
    return pd.DataFrame(rows)
        

In [ ]:
cv_results = run_cv(feature_sensors)
cv_results.pivot_table(index=['model', 'trained_on'], columns='fold', values='rmse').round(2)

In [ ]:
cv_results.groupby(['model', 'trained_on'])['rmse'].agg(['mean', 'std']).round(2)

* Training on the capped target beats the uncapped target in every one of the 5 folds, for both models, by roughly 13-14 cycles on average, far more than the fold-to-fold spread.
* Capped random forest is slightly ahead of capped linear regression (better in 4 of 5 folds, about 1 cycle on average). This is suggestive, not conclusive.
* **Decision:** train on the capped target. Random forest is the lead model; linear regression is the sanity-check baseline.

## Final Test Evaluation
Models are retrained on **all 100 training engines** (each validation fit used only 80), then scored on the last observed cycle of every test engine.

In [ ]:
# Last observed cycle of every test engine
idx = (test_001.reset_index(drop=True)
      .groupby('unit_number')['time_in_cycles'].idxmax()
      .to_numpy())
y_last = RUL_001['RUL'].to_numpy()
assert (test_001.loc[idx, 'RUL'].to_numpy() == y_last).all()

In [ ]:
# Retrain every configuration on all 100 engines, predict the last cycles
X_train = train_001[feature_sensors]
X_last = test_001.loc[idx, feature_sensors]
targets = {'uncapped': train_001['RUL'], 'capped': train_001['RUL'].clip(upper=cap)}
results = {}

for target_name, y_tr in targets.items():
    for model_name, model in make_models().items():
        model.fit(X_train, y_tr)
        results[f'{model_name} trained {target_name}'] = model.predict(X_last)

In [ ]:
# Evaluate against the official targets
for name, p in results.items():
    rmse = np.sqrt(mean_squared_error(y_last, p))
    print(f'{name}: RMSE {rmse:.2f} (÷std {rmse / y_last.std():.2f})')

In [ ]:
print(f'{(test_001['RUL']>cap).mean():.0%} of all test rows have true RUL above {cap}, while \
{(y_last>cap).sum()} of {len(y_last)} last_cycle targets do.')